In [2]:
import pandas as pd

### Importing Election2024 dataset

In [3]:
elec2024df = pd.read_parquet("data/cleaned_first_two_parts_basic.parquet", engine="pyarrow")

In [4]:
elec2024df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1999950 entries, 0 to 1999949
Data columns (total 31 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   id                         string 
 1   text                       string 
 2   url                        string 
 3   epoch                      Float64
 4   media                      string 
 5   retweetedTweet             boolean
 6   retweetedTweetID           string 
 7   retweetedUserID            string 
 8   id_str                     string 
 9   lang                       string 
 10  rawContent                 string 
 11  replyCount                 Int64  
 12  retweetCount               Int64  
 13  likeCount                  Float64
 14  quoteCount                 Float64
 15  conversationId             string 
 16  conversationIdStr          string 
 17  hashtags                   string 
 18  mentionedUsers             string 
 19  links                      string 
 20  vi

#### Preprocess tweets

**Parse timestamp to dates**

In [ ]:
elec2024df['date'] = pd.to_datetime(elec2024df['date'])

**Classify if tweet is original, retweet, quoted, or reply**

In [ ]:
import numpy as np

def classify_tweet_type(row):
    # For retweets
    if str(row.get("retweetedTweet", "")).strip().lower() == "true":
        return "retweet"

    # For quotes
    if str(row.get("quotedTweet", "")).strip().lower() == "true":
        return "quote"

    in_reply_cols = [
        "in_reply_to_screen_name",
        "in_reply_to_status_id_str",
        "in_reply_to_user_id_str"
    ]

    # For replies
    for col in in_reply_cols:
        val = row.get(col, None)
        if val is not None and str(val).strip() not in ["", "nan", "None", "<NA>"]:
            return "reply"


    # TODO: Also include mentions?

    return "original"

In [7]:
elec2024df['type'] = elec2024df.apply(classify_tweet_type, axis=1)

In [8]:
elec2024df['type'].value_counts()

type
reply       1436305
original     353249
quote        210396
Name: count, dtype: int64

**Parse user metadata (get user id)**

In [17]:
import re

def extract_user_field(user_field, key):
    try:
        if pd.isna(user_field) or str(user_field).strip() in ["", "nan", "None"]:
            return None
        
        s = str(user_field)
        
        # Regex to find 'key': value or 'key': 'value'
        # Handles both numeric and string values
        pattern = rf"'{key}'\s*:\s*'?([^',}}]+)'?"
        match = re.search(pattern, s)
        
        if match:
            return match.group(1).strip()
        return None

    except Exception as e:
        print(f"Failed: {e}")
        return None

In [ ]:
# Test
sample = elec2024df['user'].dropna().iloc[0]
result = extract_user_field(sample, "id")
print("Extracted ID:", result)

Extracted ID: 942869257108455424


In [ ]:
elec2024df['userId'] = elec2024df['user'].apply(lambda x: extract_user_field(x, "id"))
elec2024df['username'] = elec2024df['user'].apply(lambda x: extract_user_field(x, "username"))

print(elec2024df[['id', 'userId', 'username']].head())

                    id               userId          username
0  1801041792923578484   942869257108455424      B. G. Dalena
1  1801041792630227173  1461100431329796100  Brandon Holloway
2  1801041792592224521  1655734665955737600              John
3  1801041791463866688  1771777682587713536  Andrew Le Quesne
4  1801041790952231228            874708668         Joe Gaeta


**Parse `link` (URLs shared by user) column into expanded URLs**

In [21]:
import ast

def parse_links(value):
    try:
        if pd.isna(value) or str(value).strip() in ["[]", "", "nan", "None"]:
            return []
        items = ast.literal_eval(value) if isinstance(value, str) else value
        if isinstance(items, list):
            return [
                item["expanded_url"].lower()
                for item in items
                if isinstance(item, dict) and item.get("expanded_url")
            ]
        return []
    except:
        return []

In [22]:
elec2024df['expanded_urls_list'] = elec2024df['links'].apply(parse_links)

#### Compute exposure & adoption

In [ ]:
# TODO: Filter to target users only (shared >10 times)

In [ ]:
# TODO: Filter to users who shared >10 URLs

**Compute all adoptions**

In [ ]:
# TODO: Compile all URL sharing activity (user, urls shared, date)

**Compute first adoptions**

In [ ]:
# TODO: Get earliest date from adoptions using timestamp

**Compute first interactions**

In [ ]:
# TODO: First time each user interacted with the referenced user (via retweet, quote, or reply)
# NOTE: Retweet table is null while there are no quote user info, so only reply is available

**Compute exposure**

In [ ]:
# TODO: Find if user has interacted with source user after first interaction

**Compute adoption**

In [ ]:
# TODO: Check if target user adopted those URLs from source user or not

### IAR & SAR Computation

In [ ]:
# adopted_urls = total unique URLs the user ever posted
# exposure_events = total exposure events across all URLs (get sum)

# TODO: Count exposure events for URLs that the user did adopt

In [ ]:
# TODO: Count number of unique URLs adopted that had prior exposure

**Compute IAR**

In [ ]:
# IAR: of all exposure events, how often did the user adopt?

**Compute SAR**

In [ ]:
# SAR: fraction of adopted URLs that had NO prior network exposure